In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import string
import os
from nltk.tokenize import word_tokenize
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

c:\Users\schue\miniconda3\envs\text_mining_home_assignment_2\Lib\site-packages\torch\_subclasses\functional_tensor.py:276: UserWarning: Failed to initialize NumPy: DLL load failed while importing _multiarray_umath: The specified module could not be found. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# Hyperparameters

EMBEDDING_DIMENSION = 128
LR = 0.001
EPOCHS = 2
BATCH_SIZE = 32

## 3 Sequence-to-Sequence Modeling

### Data Preprocessing

In [4]:
data = pd.read_csv('topical_chat_pairs.csv', sep='\t')
data.head()

,id,conversation_id,message,answer
0,0,1,Are you a fan of Google or Microsoft?,Both are excellent technology they are helpfu...
1,1,1,Both are excellent technology they are helpfu...,"I'm not a huge fan of Google, but I use it a..."
2,2,1,"I'm not a huge fan of Google, but I use it a...",Google provides online related services and p...
3,3,1,Google provides online related services and p...,"Yeah, their services are good. I'm just not a..."
4,4,1,"Yeah, their services are good. I'm just not a...",Google is leading the alphabet subsidiary and...


In [5]:
word2index = {}
index2word = {}

word2index['<SOS>'] = 0
word2index['<EOS>'] = 1
word2index['<PAD>'] = 2
index2word[0] = '<SOS>'
index2word[1] = '<EOS>'
index2word[2] = '<PAD>'

In [6]:
lenghts = []
unique_words = set()
for text in data['message']:
    tokens = word_tokenize(text.strip().lower())
    lenghts.append(len(tokens))
    for word in tokens:
        unique_words.add(word)
for text in data['answer']:
    tokens = word_tokenize(text.strip().lower())
    lenghts.append(len(tokens))
    for word in tokens:
        unique_words.add(word)
print(f'Number of unique words: {len(unique_words)}')
print(f'{sorted(list(unique_words))[:5]}')
AVERAGE_LENGTH = round(np.mean(lenghts))
print(f'Average length of sentences: {AVERAGE_LENGTH}')

Number of unique words: 41112
['!', '#', '$', '%', '&']
Average length of sentences: 23


In [7]:
for word in unique_words:
    index = len(word2index)
    word2index[word] = index
    index2word[index] = word
print(f'Number of words in vocabulary: {len(word2index)}')

Number of words in vocabulary: 41115


In [8]:
def collate_fn(batch):
    messages, answers = zip(*batch)
    preprocessed_messages = []
    for message in messages:
        message = word_tokenize(message.strip().lower())
        preprocessed_message = []
        preprocessed_message.append(word2index['<SOS>'])
        for word in message[:AVERAGE_LENGTH]:
            preprocessed_message.append(word2index[word])
        preprocessed_message.append(word2index['<EOS>'])
        if len(preprocessed_message) < AVERAGE_LENGTH + 2:  # +2 for <SOS> and <EOS>
            preprocessed_message += [word2index['<PAD>']] * (AVERAGE_LENGTH + 2 - len(preprocessed_message))
        preprocessed_messages.append(preprocessed_message)

    max_length = max(len(text) for text in answers)
    preprocessed_answers = []
    for answer in answers:
        answer = word_tokenize(answer.strip().lower())
        preprocessed_answer = []
        preprocessed_answer.append(word2index['<SOS>'])
        for word in answer[:AVERAGE_LENGTH]:
            preprocessed_answer.append(word2index[word])
        preprocessed_answer.append(word2index['<EOS>'])
        if len(preprocessed_answer) < AVERAGE_LENGTH + 2:  # +2 for <SOS> and <EOS>
            preprocessed_answer += [word2index['<PAD>']] * (AVERAGE_LENGTH + 2 - len(preprocessed_answer))
        preprocessed_answers.append(preprocessed_answer)

    preprocessed_messages = torch.tensor(preprocessed_messages, dtype=torch.long)
    preprocessed_answers = torch.tensor(preprocessed_answers, dtype=torch.long)
    return preprocessed_messages, preprocessed_answers

In [9]:
class SequenceDataset(Dataset):
    def __init__(self, messages, answers):
        if len(messages) != len(answers):
            raise ValueError("Messages and answers must have the same length.")
        if not isinstance(messages, list) or not isinstance(answers, list):
            try:
                messages = messages.tolist()
                answers = answers.tolist()
            except AttributeError:
                raise TypeError("Messages and answers must be convertible to lists.")
        self.messages = messages
        self.answers = answers

    def __len__(self):
        return len(self.messages)

    def __getitem__(self, idx):
        return self.messages[idx], self.answers[idx]

In [10]:
dataset = SequenceDataset(data['message'], data['answer'])

In [25]:
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, drop_last=True)

In [26]:
for messages, answers in train_loader:
    print(f'Messages batch shape: {messages.shape}')
    print(f'Answers batch shape: {answers.shape}')
    for i in range(4):
        print(f'Message {i}: {messages[i]}')
        print(f'Answer {i}: {answers[i]}')
    break

Messages batch shape: torch.Size([32, 25])
Answers batch shape: torch.Size([32, 25])
Message 0: tensor([    0, 14488, 26793,  8208, 21835, 11288, 11032, 14615,  8452, 35179,
        26054, 12337, 21835,  4049, 20857,  5972,  8452,     1,     2,     2,
            2,     2,     2,     2,     2])
Answer 0: tensor([    0, 41096, 38160,  6576, 26606, 35386, 24554,  1374, 36623, 39231,
        10843,  2711, 39231, 24790, 15414,  4831, 41005, 20943, 31658, 21835,
        36688, 13317,  8208,  8452,     1])
Message 1: tensor([    0, 14488, 32600, 38160, 20527,  9131,  9140, 40988, 21835,  8067,
        20857, 17996,  7019, 11032, 22273,  8021,  4938,  8452, 18164, 38094,
        24554,  6887, 33397, 22273,     1])
Answer 1: tensor([    0, 14488, 14216, 16531, 11032,  7838, 23335, 14488, 15777, 10859,
        22273, 32130,  8452,     1,     2,     2,     2,     2,     2,     2,
            2,     2,     2,     2,     2])
Message 2: tensor([    0, 23496,  8452, 10859, 13055, 24554,   506,  8452

### Model

In [27]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)

    def forward(self, input):
        embedded = self.embedding(input)
        h0 = torch.zeros(1, BATCH_SIZE, self.hidden_size, device=embedded.device)
        _, output = self.gru(embedded, h0)
        return output

In [28]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.relu = nn.ReLU()
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.linear = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=2)

    def forward(self, input, hidden):
        embedded = self.embedding(input)
        embedded = self.relu(embedded)
        embedded = embedded.unsqueeze(0)
        # print(f'Input shape: {input.shape}, Embedded shape: {embedded.shape}')
        # print(f"Hidden state shape: {hidden.shape}")
        output, last_hidden = self.gru(embedded, hidden)
        # print(f'Output shape: {output.shape}, Last hidden shape: {last_hidden.shape}')
        output = self.linear(output)
        # print(f'Last hidden shape after linear: {output.shape}')
        output = self.softmax(output)
        # print(f'Last hidden shape after softmax: {output.shape}')
        output = output.squeeze(0)
        # print(f'Output shape after squeeze: {output.shape}')
        return output, last_hidden

In [29]:
encoder = EncoderRNN(len(word2index), EMBEDDING_DIMENSION).to(DEVICE)
decoder = DecoderRNN(EMBEDDING_DIMENSION, len(word2index)).to(DEVICE)

encoder_optimizer = torch.optim.Adam(encoder.parameters(), lr=LR)
decoder_optimizer = torch.optim.Adam(decoder.parameters(), lr=LR)
criterion = nn.NLLLoss(ignore_index=word2index['<PAD>'])

if not os.path.exists('models'):
    os.makedirs('models')
encoder_path = 'models/encoder.pth'
decoder_path = 'models/decoder.pth'

In [30]:
def validate_model(encoder, decoder, val_loader, criterion, device=DEVICE):
    encoder.eval()
    decoder.eval()
    total_loss = 0.0

    with torch.no_grad():
        val_loader = tqdm(val_loader, desc="Validating")
        for i, (messages, answers) in enumerate(val_loader):
            messages = messages.to(device)
            answers = answers.to(device)

            encoder_output = encoder(messages)
            last_hidden = encoder_output
            # print(f'Answers shape: {answers.shape}')
            decoder_input = answers[:, 0]  # Start with the first token of the answer
            batch_loss = 0.0
            for i in range(1, answers.size(1)):
                # print(f'Decoder input shape: {decoder_input.shape}')
                output, last_hidden = decoder(decoder_input, last_hidden)
                # print(f'Decoder output shape: {output.shape}')
                # print(f'Decoder last hidden shape: {last_hidden.shape}')
                loss = criterion(output, answers[:, i])
                batch_loss += loss.item()
                decoder_input = torch.argmax(output, dim=1)  # Use the predicted token as the next input
            total_loss += batch_loss / answers.size(1)
    avg_loss = total_loss / len(val_loader)
    return avg_loss

In [ ]:
def train(encoder, decoder, train_loader, val_loader, criterion, encoder_optimizer, decoder_optimizer, epochs, device):
    average_training_loss_per_epoch = []
    validation_loss_per_epoch = []
    min_validation_loss = float('inf')
    for epoch in range(epochs):
        encoder.train()
        decoder.train()
        total_loss = 0
        train_loader = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs}')
        for i, (messages, answers) in enumerate(train_loader):
            messages, answers = messages.to(device), answers.to(device)
            encoder_optimizer.zero_grad()
            decoder_optimizer.zero_grad()

            encoder_output = encoder(messages)
            last_hidden = encoder_output
            loss = 0
            for i in range(1, answers.size(1)):
                output, last_hidden = decoder(answers[:, i-1], last_hidden)
                print(f'Output shape: {output.shape}, Answers shape: {answers[:, i].shape}')
                break
                loss += criterion(output, answers[:, i])
                # decoder_input = torch.argmax(output, dim=1) # uncomment to not use teacher forcing

            loss = loss / answers.size(1)
            loss.backward()
            encoder_optimizer.step()
            decoder_optimizer.step()

            total_loss += loss.item() / answers.size(1)
            train_loader.set_postfix({"loss": total_loss / (i + 1)})

        average_training_loss_per_epoch.append(total_loss / len(train_loader))
        validation_loss = validate_model(encoder, decoder, val_loader, criterion, device)
        validation_loss_per_epoch.append(validation_loss)
        if validation_loss < min_validation_loss:
            min_validation_loss = validation_loss
            torch.save(encoder.state_dict(), encoder_path)
            torch.save(decoder.state_dict(), decoder_path)
            print(f"Validation loss improved to {min_validation_loss:.4f}, models saved.")
        

In [ ]:
train(
    encoder=encoder,
    decoder=decoder,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    encoder_optimizer=encoder_optimizer,
    decoder_optimizer=decoder_optimizer,
    epochs=EPOCHS,
    device=DEVICE
)

Epoch 1/2:  99%|█████████▉| 4465/4493 [01:39<00:00, 44.82it/s, loss=32.9]